In [6]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

books = [
    {"title": "Python Crash Course", "author": "Eric Matthes", "category": "Programming", "available": True},
    {"title": "Clean Code", "author": "Robert C. Martin", "category": "Software Engineering", "available": False},
    {"title": "Hands-On Machine Learning", "author": "Aurélien Géron", "category": "Machine Learning", "available": True},
    {"title": "Artificial Intelligence: A Modern Approach", "author": "Stuart Russell", "category": "AI", "available": True},
    {"title": "Introduction to Algorithms", "author": "Thomas Cormen", "category": "Algorithms", "available": False},
    {"title": "Database System Concepts", "author": "Abraham Silberschatz", "category": "Database", "available": True},
    {"title": "Computer Networking", "author": "James Kurose", "category": "Networking", "available": True},
    {"title": "The Data Science Handbook", "author": "Field Cady", "category": "Data Science", "available": False},
    {"title": "Learning Web Design", "author": "Jennifer Robbins", "category": "Web Development", "available": True},
    {"title": "Atomic Habits", "author": "James Clear", "category": "Self Help", "available": True}
]

def library_agent(question):
    data = "\n".join(
        f"{b['title']} | {b['author']} | {b['category']} | {'Available' if b['available'] else 'Not Available'}"
        for b in books
    )

    prompt = f"""
You are a simple CMRIT Library Assistant.
Answer the user's question ONLY using the library data below.
Never invent books or information.

Library Data:
{data}

User Question: {question}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

while True:
    question = input("You: ")

    if question.lower() in ["exit", "quit"]:
        print("Library Agent: Goodbye!")
        break

    print("Library Agent:", library_agent(question))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Library Agent: Please provide your question. I am ready to assist you based on the CMRIT Library data.
Library Agent: Please provide your question. I am ready to assist you based on the CMRIT Library data I have.
Library Agent: Database System Concepts
Library Agent: Eric Matthes
Robert C. Martin
Aurélien Géron
Stuart Russell
Thomas Cormen
Abraham Silberschatz
James Kurose
Field Cady
Jennifer Robbins
James Clear
Library Agent: *   Eric Matthes: Python Crash Course
*   Robert C. Martin: Clean Code
Library Agent: Jennifer Robbins
Library Agent: 7
Library Agent: There are 3 books not available.
Library Agent: Please ask me a question about the library's books.
Library Agent: Please provide your question about the library! I will do my best to answer it using the available library data.
Library Agent: Please provide your question about the library. I will answer it using only the available library data.
Library Agent: Please provide your question. I can help you find books from the CMRIT L

KeyboardInterrupt: 

# Improved code 


In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found. Please set it in your .env file.")

client = genai.Client(api_key=api_key)

# 1. Persistent Data Layer
DATA_FILE = Path("books.json")
if not DATA_FILE.exists():
    initial_books = [
        {"title": "Python Crash Course", "author": "Eric Matthes", "category": "Programming", "available": True},
        {"title": "Clean Code", "author": "Robert C. Martin", "category": "Software Engineering", "available": False},
        {"title": "Hands-On Machine Learning", "author": "Aurélien Géron", "category": "Machine Learning", "available": True},
        {"title": "Artificial Intelligence: A Modern Approach", "author": "Stuart Russell", "category": "AI", "available": True},
        {"title": "Introduction to Algorithms", "author": "Thomas Cormen", "category": "Algorithms", "available": False},
        {"title": "Database System Concepts", "author": "Abraham Silberschatz", "category": "Database", "available": True},
        {"title": "Computer Networking", "author": "James Kurose", "category": "Networking", "available": True},
        {"title": "The Data Science Handbook", "author": "Field Cady", "category": "Data Science", "available": False},
        {"title": "Learning Web Design", "author": "Jennifer Robbins", "category": "Web Development", "available": True},
        {"title": "Atomic Habits", "author": "James Clear", "category": "Self Help", "available": True}
    ]
    DATA_FILE.write_text(json.dumps(initial_books, indent=2))

def load_books() -> list[dict]:
    return json.loads(DATA_FILE.read_text())

def save_books(books: list[dict]):
    DATA_FILE.write_text(json.dumps(books, indent=2))

# 2. Agent Tools
def search_books(query: str) -> str:
    """Search for books by title, author, or category."""
    books = load_books()
    q = query.lower()
    matches = [
        b for b in books 
        if q in b["title"].lower() or q in b["author"].lower() or q in b["category"].lower()
    ]
    if not matches:
        return f"No books found matching '{query}'."
    return json.dumps(matches, indent=2)

def borrow_book(title: str) -> str:
    """Borrow a book by its title. Updates the book's availability to False."""
    books = load_books()
    for b in books:
        if b["title"].lower() == title.lower():
            if not b["available"]:
                return f"'{b['title']}' is currently already borrowed and unavailable."
            b["available"] = False
            save_books(books)
            return f"Success: You have borrowed '{b['title']}'."
    return f"Book '{title}' was not found in the catalog."

def return_book(title: str) -> str:
    """Return a borrowed book by its title. Updates availability to True."""
    books = load_books()
    for b in books:
        if b["title"].lower() == title.lower():
            if b["available"]:
                return f"'{b['title']}' is already marked as available."
            b["available"] = True
            save_books(books)
            return f"Success: You have returned '{b['title']}'."
    return f"Book '{title}' was not found in the catalog."

# 3. Initialize Agent Chat Session
system_prompt = """
You are the CMRIT Library Agent.
You assist students with finding books, checking availability, borrowing, and returning books.
Rules:
- Always use your tools to query or update book records.
- Never invent books or state information that isn't returned by your tools.
- Be polite, concise, and helpful.
"""

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        tools=[search_books, borrow_book, return_book],
        temperature=0.2
    )
)

# 4. Multi-turn Interaction
def ask_library_agent(user_query: str) -> str:
    response = chat.send_message(user_query)
    return response.text

